# RegNet（CIFAR100を用いた画像認識）

---
## 目的
畳み込みニューラルネットワークのモデルとして，RegNetを用いてCIFAR100データセットの100クラス物体認識を行う．RegNetは，1つのネットワーク構造を人手やNASで求めるのではなく，良いネットワーク群に共通する構造規則を少数のパラメータで表した「設計空間（Design Space）」を提案したモデルであり，その設計思想と，設計空間パラメータからステージ構成を計算する方法について理解する．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time

import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchsummary

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
CIFAR100データセットを読み込みます．CIFAR100はCIFAR10と同じく32×32ピクセルのカラー画像データセットですが，クラス数が100（1クラスあたり学習用500枚，評価用100枚）とより細かいクラス分けになっています．

学習用データに対しては，`RandomCrop`と`RandomHorizontalFlip`によるData Augmentationを適用します．

In [ ]:
transform_train = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                       transforms.RandomHorizontalFlip(),
                                       transforms.ToTensor()])
transform_test = transforms.Compose([transforms.ToTensor()])

train_data = torchvision.datasets.CIFAR100(root="./", train=True, transform=transform_train, download=True)
test_data = torchvision.datasets.CIFAR100(root="./", train=False, transform=transform_test, download=True)

print(len(train_data), len(test_data))

## RegNet
RegNet（Designing Network Design Spaces, 2020）は，1つの最適なネットワーク構造を人手で設計したり，NAS（Neural Architecture Search）で1つだけ探索したりするのではなく，「良い認識精度を示すネットワーク群に共通する構造上の規則」を見つけ出し，ネットワーク構造全体を少数のパラメータで規定する設計空間（Design Space）を提案したモデルです．

具体的には，ResNet/ResNeXt型のネットワークにおいて，各ステージ（特徴マップの解像度が共通するブロックのまとまり）のチャンネル幅・段数をランダムに変化させた大量のネットワーク集団を学習・評価したところ，精度の良いネットワーク群では，各ステージのチャンネル幅がステージのインデックスに対してほぼ線形に増加していることが分かりました．RegNetは，この経験則を初期幅$w_0$，傾き$w_a$，量子化係数$w_m$という3つのパラメータで表される線形関数として定式化し，これを丸め込む（量子化する）ことでステージごとの幅と段数を自動的に決定します．

RegNetの探索対象は，ネットワークの重みそのものだけでなく，$w_0$，$w_a$，$w_m$，グループ幅$g$，ボトルネック比$b$といった少数の設計空間パラメータである点が，個々のネットワーク構造を直接探索する従来のNASとは異なります．本ノートブックでは，論文の考え方に沿って，これらのパラメータからステージ構成（各ステージのチャンネル数・ブロック数）を計算する関数を実装したのち，$w_0$，$w_a$，$w_m$を1組具体的に指定した，設計空間中の1つのネットワーク（1つのインスタンス）を構築します．多数のネットワークを学習して比較する探索処理自体は行いません．

In [ ]:
def generate_regnet_stages(depth, w0, wa, wm, q=8):
    # 各ブロックiに対する目標幅を，w0を切片，waを傾きとする線形関数として求める
    block_indices = np.arange(depth)
    widths_cont = w0 + wa * block_indices

    # wmを底とする指数のどこに位置するかを丸めることで，幅を量子化する
    # （w0 * wm^0, w0 * wm^1, ... という等比数列上の値に丸め込むイメージ）
    exponents = np.round(np.log(widths_cont / w0) / np.log(wm))
    widths = w0 * np.power(wm, exponents)

    # 畳み込みのチャンネル数として扱いやすいよう，qの倍数に丸める
    widths = (np.round(widths / q) * q).astype(int)

    # 連続して同じ幅になっているブロックを1つのステージとしてまとめる
    stage_widths, stage_depths = [], []
    for w in widths:
        if len(stage_widths) > 0 and stage_widths[-1] == w:
            stage_depths[-1] += 1
        else:
            stage_widths.append(int(w))
            stage_depths.append(1)

    return stage_widths, stage_depths


# 設計空間パラメータの例：w0=32, wa=8, wm=2.0 から決まるステージ構成を確認する
stage_widths, stage_depths = generate_regnet_stages(depth=12, w0=32, wa=8, wm=2.0)
print('stage widths:', stage_widths)
print('stage depths:', stage_depths)

### X Block（グループ畳み込みBottleneck）
RegNetの各ブロックは，ResNeXtと同様にグループ畳み込みを用いたBottleneck構造で，論文中ではX Blockと呼ばれています．1×1畳み込みで入力チャンネル数をボトルネック比`bottleneck_ratio`に応じたチャンネル数に変換したのち，3×3のグループ畳み込み（`groups`はボトルネックのチャンネル数をグループ幅`group_width`で割った値）で空間方向の畳み込みを行い，再び1×1畳み込みでステージの出力チャンネル数に戻します．入力と出力で形状（チャンネル数・解像度）が異なる場合は，1×1畳み込みによるショートカットで形状を合わせてから加算します．

グループ畳み込みは，チャンネル方向にいくつかのグループへ分割し，各グループ内でのみ畳み込みを行うことで，通常の畳み込みに比べてパラメータ数と計算量を削減する工夫です．

In [ ]:
class RegXBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, group_width=8, bottleneck_ratio=1):
        super().__init__()
        bottleneck_channels = out_channels // bottleneck_ratio
        groups = bottleneck_channels // group_width  # グループ幅からグループ数を求める

        self.convs = nn.Sequential(
            nn.Conv2d(in_channels, bottleneck_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(bottleneck_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(bottleneck_channels, bottleneck_channels, kernel_size=3, stride=stride,
                      padding=1, groups=groups, bias=False),
            nn.BatchNorm2d(bottleneck_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(bottleneck_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = None

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        residual = x
        out = self.convs(x)
        if self.shortcut is not None:
            residual = self.shortcut(x)
        out += residual
        out = self.relu(out)
        return out

### RegNetの定義
`generate_regnet_stages`で計算したステージごとの幅・段数を用いて，`stem`（最初の3×3畳み込み）→3つのステージ（各ステージ内でX Blockを積み重ね，各ステージの先頭ブロックで特徴マップの解像度を半分にする）→Global Average Pooling→全結合層，という構成でRegNetを構築します．CIFAR100（32×32入力）に合わせて，ステージ数は3（ダウンサンプリングは2回）としています．

In [ ]:
class RegNet(nn.Module):
    def __init__(self, depth=12, w0=32, wa=8, wm=2.0, group_width=8, bottleneck_ratio=1, n_class=100):
        super().__init__()
        stage_widths, stage_depths = generate_regnet_stages(depth, w0, wa, wm)
        assert len(stage_widths) == 3, \
            'CIFAR版では32x32入力に合わせてダウンサンプリングを2回（3ステージ）としているため，' \
            'パラメータの組み合わせを変えてステージ数が3以外になった場合は動作しません．'

        self.stem = nn.Sequential(
            nn.Conv2d(3, stage_widths[0], kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(stage_widths[0]),
            nn.ReLU(inplace=True),
        )

        self.in_channels = stage_widths[0]
        stages = []
        for i, (width, n_blocks) in enumerate(zip(stage_widths, stage_depths)):
            stride = 1 if i == 0 else 2  # 各ステージの先頭ブロックで解像度を半分にする（最初のステージのみstride=1）
            stages.append(self._make_stage(width, n_blocks, stride, group_width, bottleneck_ratio))
        self.stages = nn.Sequential(*stages)

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(stage_widths[-1], n_class)

    def _make_stage(self, width, n_blocks, stride, group_width, bottleneck_ratio):
        layers = [RegXBlock(self.in_channels, width, stride, group_width, bottleneck_ratio)]
        self.in_channels = width
        for _ in range(n_blocks - 1):
            layers.append(RegXBlock(self.in_channels, width, 1, group_width, bottleneck_ratio))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)
        x = self.stages(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

## ネットワークの作成
定義したネットワークを作成します．`depth`，`w0`，`wa`，`wm`，`group_width`，`bottleneck_ratio`により，設計空間中のどのネットワークを使うかを指定します．`.to(device)`によりモデルのパラメータを`device`上に配置します．

最適化手法にはモーメンタムSGDを用い，学習率0.01，モーメンタム0.9，weight decayを5e-4とします．最後に`torchsummary.summary()`でネットワークの詳細情報を表示します．

In [ ]:
model = RegNet(depth=12, w0=32, wa=8, wm=2.0, group_width=8, bottleneck_ratio=1, n_class=100)
model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

torchsummary.summary(model, (3, 32, 32), device=device.type)

## 学習
読み込んだCIFAR100データセットと作成したネットワークを用いて学習を行います．ミニバッチサイズを128，学習エポック数を20とします．

In [ ]:
batch_size = 128
epoch_num = 20

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2)

criterion = nn.CrossEntropyLoss()
criterion = criterion.to(device)

model.train()

train_start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0
    count = 0

    for image, label in train_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        loss = criterion(y, label)
        model.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

    print("epoch: {}, mean loss: {}, mean accuracy: {}, elapsed time: {}".format(
        epoch, sum_loss / len(train_loader), count.item() / len(train_data), time() - train_start))

## テスト
学習したネットワークモデルを用いて評価を行います．

In [ ]:
test_loader = torch.utils.data.DataLoader(test_data, batch_size=100, shuffle=False)

model.eval()

count = 0
with torch.no_grad():
    for image, label in test_loader:
        image = image.to(device)
        label = label.to(device)

        y = model(image)

        pred = torch.argmax(y, dim=1)
        count += torch.sum(pred == label)

print("test accuracy: {}".format(count.item() / len(test_data)))

## オリジナル（ImageNet入力）のRegNetとCIFAR版RegNetの違い
このノートブックで実装したRegNetは，ImageNetを対象とした元論文の構造をCIFAR100（32×32入力）向けに変更したものです．主な違いは次の通りです．

| | ImageNet版（原論文） | CIFAR版（本ノートブック） |
|---|---|---|
| stem | 3×3畳み込み，stride 2で一気に解像度を1/2に縮小 | 3×3畳み込み，stride 1（解像度を維持） |
| ステージ数 | 4ステージ（各ステージ先頭でstride 2のダウンサンプリング） | 3ステージ（32×32という小さな入力に合わせダウンサンプリング回数を2回に削減） |
| 入力解像度に対する総ダウンサンプリング倍率 | 32倍 | 4倍（32×32→8×8） |
| 設計空間の扱い | 多数の$(w_0, w_a, w_m, g, b, \text{depth})$の組をランダムに生成・学習し，精度の良い領域を統計的に探索（population-based search） | $(w_0, w_a, w_m, g, b, \text{depth})$を1組だけ具体的に指定し，そこから決まる1つのネットワークのみを構築・学習（探索処理自体は行わない） |
| 出力層のクラス数 | 1000 | 100（CIFAR100） |

なお，個々のブロック内部の構造（1×1→3×3グループ畳み込み→1×1のX Block）自体はImageNet版・CIFAR版で変更していません．

## 課題

1. `w0`，`wa`，`wm`を変更し，`generate_regnet_stages`が計算する各ステージの幅・段数がどのように変化するか，またモデル全体のパラメータ数がどう変わるか確認しましょう．

2. `group_width`や`bottleneck_ratio`を変更して，`resnext.ipynb`のResNeXtブロックと比較しながら，パラメータ数・認識精度への影響を確認しましょう．

3. `depth`（総ブロック数）を変更してネットワークを深く/浅くし，学習の収束の様子や認識精度がどのように変化するか比較しましょう．